# Feature Selection + PCA

## Part A: Setup + Baseline

**Load data**

In [13]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score

df = pd.read_csv('../data/interim/f1_results_interim.csv')
print('Shape:', df.shape)
df.head()

Shape: (25850, 16)


,season,round,race_name,driver_id,driver_name,constructor_id,constructor,grid,position,points,laps,status,time,fastest_lap,fastest_lap_rank,finished
0,1950,1,British Grand Prix,farina,Nino Farina,alfa,Alfa Romeo,1,1,9.0,70,Finished,2:13:23.600,NaN,NaN,True
1,1950,1,British Grand Prix,fagioli,Luigi Fagioli,alfa,Alfa Romeo,2,2,6.0,70,Finished,+2.600,NaN,NaN,True
2,1950,1,British Grand Prix,reg_parnell,Reg Parnell,alfa,Alfa Romeo,4,3,4.0,70,Finished,+52.000,NaN,NaN,True
3,1950,1,British Grand Prix,cabantous,Yves Cabantous,lago,Talbot-Lago,6,4,3.0,68,+2 Laps,NaN,NaN,NaN,False
4,1950,1,British Grand Prix,rosier,Louis Rosier,lago,Talbot-Lago,9,5,2.0,68,+2 Laps,NaN,NaN,NaN,False


**Split X and Y**

In [14]:
# Clean: grid == 0 means the driver did not start from a real grid.
df = df[df['grid'] != 0].copy()

df['constructor_encoded'] = df['constructor'].astype('category').cat.codes

features = ['grid', 'season', 'round', 'constructor_encoded']
X = df[features].to_numpy()
Y = (df['points'] > 0).astype(int).to_numpy()

print('Rows after cleaning:', len(df))
print('Feature matrix:', X.shape)
print('Class balance:', Y.mean().round(3))
print('Constructors:', df['constructor'].nunique())

Rows after cleaning: 25584
Feature matrix: (25584, 4)
Class balance: 0.329
Constructors: 199


**Note on encoding:** `X = [grid, season, round, constructor_encoded]` is followed exactly, so the constructor category is encoded as a single integer code rather than one-hot (one-hot would add ~200 columns and change the feature set). Integer coding does impose an artificial order on constructors, but since it is only one of four features and the model is regularized, this is an acceptable simplification.

**Train/test split (used consistently across all methods)**

In [15]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, stratify=Y, random_state=42
)
print('Train:', X_train.shape, 'Test:', X_test.shape)
print('Train positive rate:', Y_train.mean().round(3), '| Test positive rate:', Y_test.mean().round(3))

Train: (20467, 4) Test: (5117, 4)
Train positive rate: 0.329 | Test positive rate: 0.329


**Baseline model (all features)**

In [16]:
# Baseline: logistic regression on all 4 features, inside a pipeline
# (scaling is included for consistency with the PCA / selection methods used later)
baseline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000)),
])
baseline.fit(X_train, Y_train)

basePred = baseline.predict(X_test)
baseF1 = f1_score(Y_test, basePred)
baseAcc = accuracy_score(Y_test, basePred)
print(f'Baseline (all 4 features)  F1 = {baseF1:.4f}  Accuracy = {baseAcc:.4f}')

Baseline (all 4 features)  F1 = 0.6223  Accuracy = 0.7694


## Part B: Sequential Feature Selection